PINN Modeling

In [1]:
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from physics_loss import physics_loss
from phis_init import phi_init



class DeltaPhisPINN(nn.Module):

    epochs = 1000

    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(2, 8),
            nn.Tanh(),
            nn.Linear(8, 1)
        )

    def forward(self, Vgs, Vds):

        Vgs = Vgs.reshape(-1, 1)
        Vds = Vds.reshape(-1, 1)

        x = torch.cat(
            [Vgs, Vds],
            dim=1
        )

        return self.net(x)

Training Parts: 
1. Input definiton
2. Training Framework

In [2]:
device = "xpu" if torch.xpu.is_available() else "cpu"
dtype = torch.float32

# Parameters
T = 300.0                         # K

NA = 1e16                        # cm^-3

# Absolute permittivity of 4H-SiC
# eps_sic = eps_r * eps_0
# eps_0 = 8.854e-14 F/cm
eps_sic = 9.26 * 8.854e-14       # F/cm

Cox = 7e-8                       # F/cm^2

Vfbs0 = -1.0                     # V

Dit_mid = 2e11                   # cm^-2 eV^-1

Dit_edge = 2e13                  # cm^-2 eV^-1

sigma_it = 0.1                   # eV

Eg = 3.26                        # eV, 4H-SiC

# Approximation:
# Ec - Ei ≈ Eg / 2
Ec_minus_Ei = Eg / 2.0           # eV

# Oxide fixed charge density
# First use ideal oxide for debugging
Qox = 0.0                        # C/cm^2

# Input range defined
VGS_MIN = -10.0
VGS_MAX = 30.0

VDS_MIN = 0.0
VDS_MAX = 10.0

# Training samples number(dots number)
N_train = 500

Vgs_train = (
    VGS_MIN
    + (VGS_MAX - VGS_MIN)
    * torch.rand(
        N_train,
        1,
        dtype=dtype,
        device=device
    )
)

Vds_train = (
    VDS_MIN
    + (VDS_MAX - VDS_MIN)
    * torch.rand(
        N_train,
        1,
        dtype=dtype,
        device=device
    )
)


# Create PINN Model
Model = DeltaPhisPINN().to(
    device=device,
    dtype=dtype
)

optimizer = torch.optim.Adam(
    Model.parameters(),
    lr=1e-4
)

loss_history = []

# ============================================================
# Mini-batch training
# ============================================================

batch_size = 128

Model.train()

for ep in range(Model.epochs):

    # Randomly shuffle training indices
    permutation = torch.randperm(
        N_train,
        device=device
    )

    epoch_loss = 0.0
    batch_count = 0

    for start in range(0, N_train, batch_size):

        idx = permutation[
            start:start + batch_size
        ]

        Vgs_batch = Vgs_train[idx]
        Vds_batch = Vds_train[idx]

        optimizer.zero_grad(set_to_none=True)

        # Drain-side quasi-Fermi potential
        phi_f_batch = Vds_batch

        # Analytical initial surface potential
        phis_init_batch = phi_init(
            Vgs_batch,
            phi_f_batch,
            T,
            NA,
            eps_sic,
            Cox,
            Vfbs0,
            Dit_mid,
            Dit_edge,
            sigma_it,
            Eg,
        )

        phis_init_batch = phis_init_batch.reshape(
            -1,
            1
        )

        # Neural-network correction
        delta_phis_PINN = Model(
            Vgs_batch,
            Vds_batch
        )

        # Final PINN surface potential
        phis_PINN = (
            phis_init_batch
            + delta_phis_PINN
        )

        # Physics-informed loss
        loss = physics_loss(
            phis_PINN,
            Vgs_batch,
            Vds_batch,
            T,
            NA,
            eps_sic,
            Cox,
            Vfbs0,
            Qox,
            Dit_mid,
            Dit_edge,
            sigma_it,
            Ec_minus_Ei,
            Eg,
        )

        # Check for NaN / Inf
        if not torch.isfinite(loss):
            raise RuntimeError(
                f"Non-finite loss detected at epoch {ep}, "
                f"batch starting at {start}: {loss.item()}"
            )

        # Backpropagation
        loss.backward()

        # Update model
        optimizer.step()

        epoch_loss += loss.item()
        batch_count += 1

    epoch_loss /= batch_count
    loss_history.append(epoch_loss)

    if ep % 100 == 0:
        print(
            f"Epoch {ep:6d}/{Model.epochs}, "
            f"L_SPE = {epoch_loss:.6e}"
        )

RuntimeError: Non-finite loss detected at epoch 0, batch starting at 0: nan

Plotting curves

In [ ]:
Model.eval()

N_plot = 400


# ============================================================
# 1. Ordered Vgs for smooth curve
# ============================================================

Vgs_plot = torch.linspace(
    VGS_MIN,
    VGS_MAX,
    N_plot,
    dtype=dtype,
    device=device
).reshape(-1, 1)


# Fixed drain voltage
VDS_PLOT = 10.0

Vds_plot = torch.full_like(
    Vgs_plot,
    VDS_PLOT
)


# ============================================================
# 2. Calculate surface potential
# ============================================================

with torch.no_grad():

    # Drain-side quasi-Fermi potential
    phi_f_plot = Vds_plot


    # Analytical initial surface potential
    phis_init_plot = phi_init(
        Vgs_plot,
        phi_f_plot,
        T,
        NA,
        eps_sic,
        Cox,
        Vfbs0,
        Dit_mid,
        Dit_edge,
        sigma_it,
        Eg,
    )

    phis_init_plot = phis_init_plot.reshape(
        -1,
        1
    )


    # PINN correction
    delta_phis_plot = Model(
        Vgs_plot,
        Vds_plot
    )


    # Final surface potential
    phis_pred = (
        phis_init_plot
        + delta_phis_plot
    )


# ============================================================
# 3. Plot
# ============================================================

plt.figure(
    figsize=(6, 4)
)

plt.plot(
    Vgs_plot.cpu().numpy(),
    phis_pred.cpu().numpy(),
    linewidth=2
)

plt.xlabel(
    r"$V_{GS}$ (V)"
)

plt.ylabel(
    r"$\phi_s$ (V)"
)

plt.title(
    rf"$\phi_s$ vs $V_{{GS}}$, "
    rf"$V_{{DS}}={VDS_PLOT}\,\mathrm{{V}}$"
)

plt.grid(True)

plt.tight_layout()

plt.show()

NameError: name 'Model' is not defined